In [1]:
from sklearn.preprocessing import StandardScaler

import numpy as np
from numpy import load
import scipy
import matplotlib.pyplot as plt
from scipy.special import legendre
from numpy.linalg import svd

from time import time
from numpy import sin as sin

import math

In [2]:
import pandas as pd
scale=StandardScaler()


In [3]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from encoding import superposition,inverse_superposition

## def function


# DATA and function visualization

In [4]:
data = load('../data_prep/disease_SR.npy')


`cheking that can we retrive the graph from super position matrix`

`scaled the 12 lead data before the input inversed_scaled is thae data which is inversed scaled from the superposition matrix `

In [5]:
import numpy as np

total_samples = len(data)
valid_indices = []
removed_indices = []

print("Sanitizing dataset (checking NaNs in raw data only)...")

for i in range(total_samples):
    cc = np.asarray(data[i])

    if np.isnan(cc).any():
        removed_indices.append(i)
    else:
        valid_indices.append(i)

print("\n========== SANITIZATION SUMMARY ==========")
print(f"Total samples       : {total_samples}")
print(f"Removed (NaN in data) : {len(removed_indices)}")
print(f"Remaining valid     : {len(valid_indices)}")

Sanitizing dataset (checking NaNs in raw data only)...

========== SANITIZATION SUMMARY ==========
Total samples       : 5000
Removed (NaN in data) : 16
Remaining valid     : 4984


In [6]:
data = load('../data_prep/unq_disease_SR.npy')


In [7]:
import numpy as np
import pandas as pd

from encoding import normalize_matrix

# ==============================
# STEP 1 — SANITIZE DATASET
# ==============================

total_samples = len(data)

print("Sanitizing dataset (checking NaNs in raw data only)...")

data_array = np.array([np.asarray(data[i]) for i in range(total_samples)])

nan_mask = np.isnan(data_array).any(axis=tuple(range(1, data_array.ndim)))

valid_indices = np.where(~nan_mask)[0].tolist()
removed_indices = np.where(nan_mask)[0].tolist()

print("\n========== SANITIZATION SUMMARY ==========")
print(f"Total samples         : {total_samples}")
print(f"Removed (NaN in data) : {len(removed_indices)}")
print(f"Remaining valid       : {len(valid_indices)}")


# ==============================
# STEP 2 — FULL RECONSTRUCTION STATISTICS
# ==============================

mse_list = []
mae_list = []
max_abs_list = []
rel_list = []

print("\nComputing reconstruction statistics on cleaned dataset...")

for i, idx in enumerate(valid_indices):
    if i % max(1, len(valid_indices) // 20) == 0:
        print(f"  Evaluating invertibility... {i}/{len(valid_indices)}", end="\r")

    cc = data_array[idx]

    sup = superposition(cc, n=5000)
    inversed_scaled = inverse_superposition(sup)
    # apply same preprocessing as superposition does internally
    cc_normalized = normalize_matrix(cc)
    scaler = StandardScaler()
    cc_scaled = scaler.fit_transform(cc_normalized)  # this is what inverse returns


    # Defensive shape check
    if inversed_scaled.shape != cc_scaled.shape:
        raise ValueError(f"Shape mismatch at index {idx}: "
                         f"{inversed_scaled.shape} vs {cc_scaled.shape}")

    error = inversed_scaled - cc_scaled  # Use scaled version for error calculation

    mse_list.append(np.mean(error**2))
    mae_list.append(np.mean(np.abs(error)))
    max_abs_list.append(np.max(np.abs(error)))

    # Relative error (L2 norm)
    rel_error = np.linalg.norm(error) / (np.linalg.norm(cc_scaled) + 1e-12)
    rel_list.append(rel_error)
    if i == 1 or i==2 or i==3:
        print(f"  result of invertibility {i}/{len(valid_indices)} ✓")
        print(f"    MSE: {mse_list[-1]:.6e}")
        print(f"    MAE: {mae_list[-1]:.6e}")
        print(f"    Max Abs Error: {max_abs_list[-1]:.6e}")
        print(f"    Relative Error: {rel_error:.6e}")


    

print(f"  Evaluating invertibility... {len(valid_indices)}/{len(valid_indices)} ✓")

# Convert to arrays
mse_array = np.array(mse_list)
mae_array = np.array(mae_list)
max_array = np.array(max_abs_list)
rel_array = np.array(rel_list)


# ==============================
# STEP 3 — PRINT AGGREGATE RESULTS
# ==============================

print("\n========== FULL CLEAN TEST SET RESULTS ==========")
print(f"Samples evaluated : {len(valid_indices)}")
print(f"MSE  Mean ± Std : {mse_array.mean():.6e} ± {mse_array.std():.6e}")
print(f"MAE  Mean ± Std : {mae_array.mean():.6e} ± {mae_array.std():.6e}")
print(f"Worst-case Max Absolute Error : {max_array.max():.6e}")
print(f"Relative Error Mean ± Std : {rel_array.mean():.6e} ± {rel_array.std():.6e}")
print(f"Worst-case Relative Error : {rel_array.max():.6e}")
print(f"95th percentile MAE : {np.percentile(mae_array,95):.6e}")
print(f"99th percentile MAE : {np.percentile(mae_array,99):.6e}")


# ==============================
# STEP 4 — SAVE AGGREGATE CSV
# ==============================

results_dict = {
    "total_samples": total_samples,
    "valid_samples": len(valid_indices),
    "removed_samples": len(removed_indices),

    "mse_mean": mse_array.mean(),
    "mse_std": mse_array.std(),

    "mae_mean": mae_array.mean(),
    "mae_std": mae_array.std(),

    "max_absolute_error_worst": max_array.max(),

    "relative_error_mean": rel_array.mean(),
    "relative_error_std": rel_array.std(),
    "relative_error_worst": rel_array.max(),

    "mae_95_percentile": np.percentile(mae_array, 95),
    "mae_99_percentile": np.percentile(mae_array, 99),
}

results_df = pd.DataFrame([results_dict])
results_df.to_csv("result_of_SR.csv", index=False)

print("\nAggregate results saved to: result_of_SR.csv")


# ==============================
# STEP 5 — SAVE PER-SAMPLE ERRORS
# ==============================

per_sample_df = pd.DataFrame({
    "sample_index": valid_indices,
    "mse": mse_array,
    "mae": mae_array,
    "max_absolute_error": max_array,
    "relative_error": rel_array
})

per_sample_df.to_csv("result_of_SR_all_samples.csv", index=False)

print("Per-sample errors saved to: result_of_SR_all_samples.csv")

Sanitizing dataset (checking NaNs in raw data only)...

========== SANITIZATION SUMMARY ==========
Total samples         : 5000
Removed (NaN in data) : 12
Remaining valid       : 4988

Computing reconstruction statistics on cleaned dataset...
  result of invertibility 1/4988 ✓88
    MSE: 7.723271e-05
    MAE: 5.794711e-03
    Max Abs Error: 6.886652e-02
    Relative Error: 8.788214e-03
  result of invertibility 2/4988 ✓
    MSE: 5.608374e-05
    MAE: 4.664732e-03
    Max Abs Error: 8.354614e-02
    Relative Error: 7.488908e-03
  result of invertibility 3/4988 ✓
    MSE: 7.157517e-05
    MAE: 4.735209e-03
    Max Abs Error: 6.714797e-02
    Relative Error: 8.460211e-03
  Evaluating invertibility... 4988/4988 ✓

========== FULL CLEAN TEST SET RESULTS ==========
Samples evaluated : 4988
MSE  Mean ± Std : 5.813572e-05 ± 1.403182e-05
MAE  Mean ± Std : 4.592181e-03 ± 7.633798e-04
Worst-case Max Absolute Error : 2.370931e-01
Relative Error Mean ± Std : 7.582892e-03 ± 9.463022e-04
Worst-case R

In [19]:
# create a function take np array and index to calculate all error
def evaluate_invertibility(data_array, index):
    cc = data_array[index]
    sup = superposition(cc, n=5000)
    inversed_scaled = inverse_superposition(sup)
    # apply same preprocessing as superposition does internally
    cc_normalized = normalize_matrix(cc)
    scaler = StandardScaler()
    cc_scaled = scaler.fit_transform(cc_normalized)  # this is what inverse returns

    error = inversed_scaled - cc_scaled  # Use scaled version for error calculation

    mse = np.mean(error**2)
    mae = np.mean(np.abs(error))
    max_abs_error = np.max(np.abs(error))
    relative_error = np.linalg.norm(error) / (np.linalg.norm(cc_scaled))

    return {
        "mse": mse,
        "mae": mae,
        "max_absolute_error": max_abs_error,
        "relative_error": relative_error
    }
data= load('../data_prep/disease_SR.npy')
evaluate_invertibility(data, 0)

{'mse': np.float64(6.769012671895634e-05),
 'mae': np.float64(0.005568927547399405),
 'max_absolute_error': np.float64(0.056358005024204516),
 'relative_error': np.float64(0.008227400969866265)}

In [8]:
import numpy as np
import pandas as pd

# ==============================
# CONFIGURATION
# ==============================

DATASETS = {
    "ST": "../data_prep/disease_ST.npy",
    "SB": "../data_prep/disease_SB.npy",
}

N_SUPERPOSITION = 5000


# ==============================
# CORE EVALUATION FUNCTION
# ==============================

def evaluate_dataset(name, path):
    print(f"\n{'='*50}")
    print(f"  Processing dataset: {name}")
    print(f"{'='*50}")

    # --- Load ---
    raw = np.load(path, allow_pickle=True)

    # --- Sanitize ---
    print("Sanitizing (checking NaNs)...")
    data_array = np.array([np.asarray(raw[i]) for i in range(len(raw))])
    nan_mask = np.isnan(data_array).any(axis=tuple(range(1, data_array.ndim)))

    valid_indices   = np.where(~nan_mask)[0].tolist()
    removed_indices = np.where( nan_mask)[0].tolist()

    print(f"  Total    : {len(raw)}")
    print(f"  Removed  : {len(removed_indices)}")
    print(f"  Valid    : {len(valid_indices)}")

    # --- Reconstruct & Evaluate ---
    print("Evaluating invertibility...")
    mse_list, mae_list, max_abs_list, rel_list = [], [], [], []

    for i, idx in enumerate(valid_indices):
        if i % max(1, len(valid_indices) // 20) == 0:
            print(f"  {i}/{len(valid_indices)}", end="\r")

        cc = data_array[idx]

        sup = superposition(cc, n=5000)
        inversed_scaled = inverse_superposition(sup)
        # apply same preprocessing as superposition does internally
        cc_normalized = normalize_matrix(cc)
        scaler = StandardScaler()
        cc_scaled = scaler.fit_transform(cc_normalized)  # this is what inverse returns


        # Defensive shape check
        if inversed_scaled.shape != cc_scaled.shape:
            raise ValueError(f"Shape mismatch at index {idx}: "
                            f"{inversed_scaled.shape} vs {cc_scaled.shape}")

        error = inversed_scaled - cc_scaled  # Use scaled version for error calculation

        mse_list.append(np.mean(error**2))
        mae_list.append(np.mean(np.abs(error)))
        max_abs_list.append(np.max(np.abs(error)))

        # Relative error (L2 norm)
        rel_error = np.linalg.norm(error) / (np.linalg.norm(cc_scaled) + 1e-12)
        rel_list.append(rel_error)

    print(f"  {len(valid_indices)}/{len(valid_indices)} ✓            ")

    mse_array = np.array(mse_list)
    mae_array = np.array(mae_list)
    max_array = np.array(max_abs_list)
    rel_array = np.array(rel_list)

    # --- Print Summary ---
    print(f"\n--- Results: {name} ---")
    print(f"  Samples evaluated         : {len(valid_indices)}")
    print(f"  MSE  Mean ± Std           : {mse_array.mean():.6e} ± {mse_array.std():.6e}")
    print(f"  MAE  Mean ± Std           : {mae_array.mean():.6e} ± {mae_array.std():.6e}")
    print(f"  Worst-case Max Abs Error  : {max_array.max():.6e}")
    print(f"  Relative Error Mean ± Std : {rel_array.mean():.6e} ± {rel_array.std():.6e}")
    print(f"  Worst-case Relative Error : {rel_array.max():.6e}")
    print(f"  95th percentile MAE       : {np.percentile(mae_array, 95):.6e}")
    print(f"  99th percentile MAE       : {np.percentile(mae_array, 99):.6e}")

    # --- Save Aggregate CSV ---
    agg_df = pd.DataFrame([{
        "dataset":                   name,
        "total_samples":             len(raw),
        "valid_samples":             len(valid_indices),
        "removed_samples":           len(removed_indices),
        "mse_mean":                  mse_array.mean(),
        "mse_std":                   mse_array.std(),
        "mae_mean":                  mae_array.mean(),
        "mae_std":                   mae_array.std(),
        "max_absolute_error_worst":  max_array.max(),
        "relative_error_mean":       rel_array.mean(),
        "relative_error_std":        rel_array.std(),
        "relative_error_worst":      rel_array.max(),
        "mae_95_percentile":         np.percentile(mae_array, 95),
        "mae_99_percentile":         np.percentile(mae_array, 99),
    }])
    agg_path = f"result_of_typ2_{name}.csv"
    agg_df.to_csv(agg_path, index=False)
    print(f"  Aggregate results saved to : {agg_path}")

    # --- Save Per-Sample CSV ---
    per_sample_df = pd.DataFrame({
        "sample_index":     valid_indices,
        "mse":              mse_array,
        "mae":              mae_array,
        "max_absolute_error": max_array,
        "relative_error":   rel_array,
    })
    per_path = f"result_of_typ2_{name}_all_samples.csv"
    per_sample_df.to_csv(per_path, index=False)
    print(f"  Per-sample errors saved to : {per_path}")


# ==============================
# RUN
# ==============================

for name, path in DATASETS.items():
    evaluate_dataset(name, path)

print("\nAll datasets processed.")


  Processing dataset: ST
Sanitizing (checking NaNs)...
  Total    : 5000
  Removed  : 13
  Valid    : 4987
Evaluating invertibility...
  4987/4987 ✓            

--- Results: ST ---
  Samples evaluated         : 4987
  MSE  Mean ± Std           : 5.108763e-05 ± 1.835962e-05
  MAE  Mean ± Std           : 4.188832e-03 ± 9.900367e-04
  Worst-case Max Abs Error  : 3.305994e-01
  Relative Error Mean ± Std : 7.077287e-03 ± 1.292144e-03
  Worst-case Relative Error : 1.024100e-02
  95th percentile MAE       : 5.505724e-03
  99th percentile MAE       : 6.033104e-03
  Aggregate results saved to : result_of_typ2_ST.csv
  Per-sample errors saved to : result_of_typ2_ST_all_samples.csv

  Processing dataset: SB
Sanitizing (checking NaNs)...
  Total    : 5000
  Removed  : 4
  Valid    : 4996
Evaluating invertibility...
  4996/4996 ✓            

--- Results: SB ---
  Samples evaluated         : 4996
  MSE  Mean ± Std           : 5.642305e-05 ± 1.273472e-05
  MAE  Mean ± Std           : 4.347605e-03 